## Word Embeddings
The Python `gensim` package provides word embeddings for *Word2Vec* and *GloVe*. The `tranformers` package allows the use of word embeddings from BERT and other transformer models.

In [ ]:
import gensim.downloader as api
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from transformers import BertTokenizer, BertModel
import torch

In [ ]:
# Retrieve information about all available models
info_dict = api.info()

# Extract the list of models under the 'models' key
available_models = info_dict['models'].keys()

# Print the available models
for model in available_models:
    print(model)

In [ ]:
def plot_pca(model, filename):
    # Words of interest and their pairs
    words = ['man', 'woman', 'king', 'queen', 'paris', 'france', 'london', 'england', 'apple', 'orange', 'car', 'bicycle']
    pairs = [('man', 'king'), ('woman', 'queen'), ('paris', 'france'), ('london', 'england'), ('apple', 'orange'), ('car', 'bicycle')]

    # Assign different colors for different groups of pairs
    colors = {'man': 'blue', 'king': 'blue', 'woman': 'green', 'queen': 'green', 
              'paris': 'red', 'france': 'red', 'london': 'purple', 'england': 'purple',
              'apple': 'orange', 'orange': 'orange', 'car': 'cyan', 'bicycle': 'cyan'}

    # Extract the word vectors
    word_vectors = [model[word] for word in words]
    

    # PCA for dimensionality reduction to 3 dimensions
    pca = PCA(n_components=3)
    word_vectors_3d = pca.fit_transform(word_vectors)
    print(word_vectors_3d)

    # 3D Plotting
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot the words
    word_coords = {word: word_vectors_3d[i] for i, word in enumerate(words)}
    for word, coord in word_coords.items():
        ax.scatter(*coord, s=50, color=colors[word], label=word)
        ax.text(*coord, word, size=10, zorder=1, color='black')

    # Draw lines between pairs with matching colors
    for word1, word2 in pairs:
        point1 = word_coords[word1]
        point2 = word_coords[word2]
        line_color = colors[word1]  # Assuming both words in the pair have the same color
        ax.plot([point1[0], point2[0]], [point1[1], point2[1]], [point1[2], point2[2]], color=line_color)

    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.set_zlabel('Component 3')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

### Word2Vec

In [ ]:
word2vec_model = api.load('word2vec-google-news-300')

In [ ]:
# Get vector for a word
print(word2vec_model['computer'])

In [ ]:
# Find similar words
print(word2vec_model.most_similar('computer'))

In [ ]:
word2vec_filename = 'Word2vec3D.png'

In [ ]:
plot_pca(word2vec_model, word2vec_filename)

### GloVe

In [ ]:
glove_model = api.load('glove-wiki-gigaword-300')

In [ ]:
# Get vector for a word
print(glove_model['computer'])

In [ ]:
# Find similar words
print(glove_model.most_similar('computer'))

In [ ]:
glove_filename = 'GloVe3D.png'

In [ ]:
plot_pca(glove_model, glove_filename)

### BERT

In [ ]:
def get_bert_embeddings(word_list):
    # Load pre-trained model tokenizer (vocabulary)
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    # Load pre-trained model
    model = BertModel.from_pretrained('bert-base-uncased')

    word_embeddings = {}

    for word in word_list:
        # Encode text
        encoded_input = tokenizer(word, return_tensors='pt')

        # Forward pass, get hidden states
        with torch.no_grad():
            outputs = model(**encoded_input)

        # Get the embeddings of the first token ([CLS] token)
        embeddings = outputs.last_hidden_state[:, 0, :]
        word_embeddings[word] = embeddings.detach().numpy().tolist()[0]

    return word_embeddings

In [ ]:
words = ['man', 'woman', 'king', 'queen', 'paris', 'france', 'london', 'england', 'apple', 'orange', 'car', 'bicycle']
bert_embeddings = get_bert_embeddings(words)

In [ ]:
# Print embeddings for each word
for word, emb in bert_embeddings.items():
    print(f"Embedding for {word}: {emb}")

In [ ]:
bert_filename = 'BERT3D.png'

In [ ]:
plot_pca(bert_embeddings, bert_filename)